# OP-10 · Vérification du multi-modèle, brut et calibré

**Notebook opérationnel** — à exécuter après OP-09, à chaque cycle.

| | |
|---|---|
| Étape du workflow | E6 (vérification de la combinaison, Draft §3.4) |
| Entrées | hindcasts C3S du cycle, registre du cycle, synthèses de skill des modèles, observations CHIRPS |
| Sorties | `OUTPUTS_OSF/skill/<cycle>/{raw,calibrated}/` sous les modèles `mme_equal`, `mme_sqrt_members`, `mme_skill` : `netcdf/`, `figures/`, `diagrams/`, et leurs lignes dans les synthèses |
| Durée indicative | ≈ 12 min par période en brut, ≈ 45 min par période en calibré (8 modèles, 3 schémas) : plusieurs heures pour tous les mois et toutes les saisons |

Le multi-modèle est vérifié **exactement comme un modèle** (D48) : mêmes produits, mêmes métriques,
même LOYO, mêmes cartes et mêmes diagrammes de fiabilité et ROC (en R, pour le schéma à poids
égaux), pour chaque schéma, en brut et en calibré. En calibré, chaque modèle est réajusté en LOYO
avec sa méthode recommandée — c'est ce qui coûte. Pour les terciles, la carte du **GROC**
accompagne celle du RPSS.

Les poids du schéma skill sont ceux de tout le hindcast : sa vérification est légèrement
optimiste. Les lignes `mme` entrent dans les synthèses, jamais dans les registres.

## Paramètres

In [ ]:
CYCLE_CONFIG = "config/cycle_202611.yaml"
VARIABLES    = ["precip"]
SCALES       = ["month", "season"]
PERIODS      = ["all-months", "all-seasons"]   # None = sous-ensemble de développement
KINDS        = ["raw", "calibrated"]
SCHEMES      = ["equal", "sqrt_members", "skill"]
DIAGRAMMES   = True          # diagrammes de fiabilité et ROC (R), schéma à poids égaux
N_BOOT       = 300

In [ ]:
from pathlib import Path
import os
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)

from eccas_s2s.settings import load_cycle
from eccas_s2s.registry import CALIBRATION, read_register, registry_path

cfg = load_cycle(CYCLE_CONFIG)
print(f"Cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()}")
registre = read_register(cfg, CALIBRATION)
if "calibrated" in KINDS:
    if registre is None:
        raise FileNotFoundError(f"{registry_path(cfg, CALIBRATION)} absent : exécuter OP-08 d'abord")
    print(f"registre du cycle : {len(registre)} décisions, {registre.model.nunique()} modèles, "
          f"échelles {sorted(registre.scale.unique())}")

## 1. Scores du multi-modèle (netCDF, synthèses, diagrammes)

In [ ]:
from eccas_s2s.operations import skill_multimodel

sk = skill_multimodel.run(CYCLE_CONFIG, variables=VARIABLES, scales=SCALES, kinds=KINDS,
                          schemes=SCHEMES, selection=PERIODS, diagrams=DIAGRAMMES, n_boot=N_BOOT)
print(f"\nStatut : {sk.status} — {len(sk.outputs)} sortie(s)")

## 2. Cartes de skill du multi-modèle

Une carte par produit, métrique décisive et période (plus le GROC des terciles). En calibré, le
multi-modèle n'a pas de méthode propre : son niveau de méthode s'appelle `recommended`.

In [ ]:
from eccas_s2s.operations import plot_skill_raw

cartes = []
if "raw" in KINDS:
    cartes.append(plot_skill_raw.run(CYCLE_CONFIG, kind="raw", systems=["mme"],
                                     variables=VARIABLES, scales=SCALES))
if "calibrated" in KINDS:
    cartes.append(plot_skill_raw.run(CYCLE_CONFIG, kind="calibrated", systems=["mme"],
                                     variables=VARIABLES, scales=SCALES, methods=["recommended"]))
for c in cartes:
    print(f"{c.step:24s} {c.status:8s} {len(c.outputs)} carte(s)")

## 3. Le multi-modèle contre les modèles seuls

RPSS et GROC des terciles, médiane sur le masque CEEAC : chaque schéma du multi-modèle, la
médiane des modèles seuls et le meilleur d'entre eux. En calibré, chaque modèle est lu avec sa
méthode **recommandée**.

In [ ]:
from eccas_s2s.operations.skill_raw import skill_dir

reco = {(r.model, r.scale, r["product"]): r.method for _, r in registre.iterrows()} if registre is not None else {}
lignes = []
for kind in KINDS:
    name = "skill_raw_summary.csv" if kind == "raw" else "skill_calibrated_summary.csv"
    s = pd.read_csv(skill_dir(cfg, kind) / name)
    s = s[(s["product"] == "terciles") & s.metric.isin(["rpss", "groc"]) & s.scale.isin(SCALES)]
    if kind == "calibrated":
        keep = (s.system == "mme") | [reco.get((m, sc, "terciles")) == me
                                      for m, sc, me in zip(s.model, s.scale, s.method)]
        s = s[keep]
    for (scale, period, label, metric), g in s.groupby(["scale", "period", "label", "metric"]):
        seuls = g[g.system == "c3s"].set_index("model")["median"]
        ligne = {"type": kind, "échelle": scale, "période": label, "métrique": metric,
                 "modèles (médiane)": seuls.median(), "meilleur modèle": seuls.max(),
                 "lequel": seuls.idxmax() if len(seuls) else ""}
        ligne.update({f"mme {m}": v for m, v in g[g.system == "mme"].set_index("model")["median"].items()})
        lignes.append(ligne)
vue = pd.DataFrame(lignes).sort_values(["métrique", "type", "échelle", "période"])
display(vue.round(3))
if "calibrated" in KINDS:
    print("En calibré, le GROC des modèles seuls est vide quand leur méthode recommandée est une loi "
          "paramétrique (NGR, ELR, logistique) : la chaîne ne le calcule que pour les méthodes qui "
          "gardent des membres. Le multi-modèle calibré, qui combine des probabilités, en a un.")

## 4. Contrôle visuel : RPSS des terciles, première saison, poids égaux

In [ ]:
from IPython.display import Image, display
from eccas_s2s.operations.skill_raw import score_paths

for kind, sub in (("raw", "rpss"), ("calibrated", "recommended/rpss")):
    if kind not in KINDS:
        continue
    dossier = score_paths(skill_dir(cfg, kind) / "figures", "mme", "equal", VARIABLES[0],
                          "season", "terciles") / sub
    figures = sorted(dossier.glob("*.png"))
    if figures:
        print(kind, figures[0].name)
        display(Image(filename=str(figures[0]), width=520))

## 5. Traçabilité

In [ ]:
for c in [sk] + cartes:
    print(f"{c.step:24s} {c.status:8s} {c.run_dir / 'manifest.json'}")